# Antimalarial activity prediction: full pipeline
The code lives in the `.py` files in this folder (`config`, `graphs`, `splits`, `data_prep`, `descriptors`, `models`, `train`, `tune`, `train_final`, `evaluate`, `curves`, `explain`, `plots`). This notebook only imports them and runs the stages, top to bottom.

* Tuning is **resumable**: if the session stops, run the *tune* cell again.
* Run once with `QUICK = True` (a few minutes) to check everything works, then set it to `False`.
* A GPU makes this several times faster.

In [ ]:
!pip install -q shap

## 1. Settings

In [ ]:
import os, time
import pandas as pd, torch

import config, data_prep, tune, train_final, evaluate, curves, explain, plots

# ---- your three input files go in the main_data_2 folder, next to this notebook and the .py files ----
DATA        = "main_data_2/Drug_activity.csv"                         # SMILES + label columns
GRAPHS      = "main_data_2/molecule_graphs.pt"                        # your graphs (dict SMILES -> Data, or list of Data with .smiles)
DESCRIPTORS = "main_data_2/enhanced_drug_data_with_descriptors.csv"   # your descriptors (SMILES + numeric columns); None = compute with RDKit
OUT         = "."                                         # results are written here
SMILES_COL, LABEL_COL = "Smiles", "Labels"

# ---- run options ------------------------------------------------------------
QUICK = True            # True = tiny smoke test; set False for the real run
SEEDS = [42, 43, 44]    # final-training repeats (reviewers asked for repeated evaluation)
AMP   = False           # mixed precision on a GPU: faster, try True once the run works
EXPLAIN_MODEL = "gat_transformer"   # gcn | gat | gcn_transformer | gat_transformer

config.CFG.update(data_path=DATA, graph_path=GRAPHS, desc_path=DESCRIPTORS, build_missing=False, out_dir=OUT,
                  smiles_col=SMILES_COL, label_col=LABEL_COL, amp=AMP, limit=None)
EXPLAIN = dict(n=500, shap_n=100, nsamples=500, background=50)
if QUICK:
    config.apply_quick(); SEEDS = SEEDS[:2]; EXPLAIN = dict(n=40, shap_n=6, nsamples=100, background=10)
torch.set_num_threads(os.cpu_count() or 1)
print("device:", config.DEVICE, "| quick:", QUICK, "| seeds:", SEEDS)

## 2. Prepare: clean data, load graphs and descriptors, scaffold splits

In [ ]:
t0 = time.time()
data_prep.prepare()
print(f'done in {(time.time() - t0) / 60:.1f} min')

## 3. Tune: random search with 5-fold scaffold cross-validation (development set only)

In [ ]:
t0 = time.time()
tune.tune()
print(f'done in {(time.time() - t0) / 60:.1f} min')

## 4. Final training and test predictions

In [ ]:
t0 = time.time()
train_final.final(SEEDS)
print(f'done in {(time.time() - t0) / 60:.1f} min')

## 5. Report: metrics, bootstrap CIs, DeLong tests

In [ ]:
t0 = time.time()
evaluate.report()
print(f'done in {(time.time() - t0) / 60:.1f} min')

## 6. Learning curves (5-fold scaffold CV, all nine models)

In [ ]:
t0 = time.time()
curves.curves()
print(f'done in {(time.time() - t0) / 60:.1f} min')

## 7. Post-hoc explanations (feature level, not validated)

In [ ]:
t0 = time.time()
explain.explain(EXPLAIN_MODEL, EXPLAIN['n'], EXPLAIN['shap_n'], EXPLAIN['nsamples'], EXPLAIN['background'])
print(f'done in {(time.time() - t0) / 60:.1f} min')

## 8. Figures

In [ ]:
t0 = time.time()
plots.plots()
print(f'done in {(time.time() - t0) / 60:.1f} min')

## 9. Look at the results

In [ ]:
from IPython.display import Image, display
for f in ["dataset_summary", "hyperparameter_table", "cv_mean_std_table", "test_set_metrics_table",
          "bootstrap_ci_table", "seed_repeat_table", "explain_agreement", "explain_deletion_check"]:
    p = os.path.join(OUT, f + ".csv")
    if os.path.exists(p):
        print("\n" + f); display(pd.read_csv(p))
for f in ["prauc_ci", "delong_heatmap", "figure5"]:       # the other figures are PDF only (original style); open them from figures/
    p = os.path.join(OUT, "figures", f + ".png")
    if os.path.exists(p):
        display(Image(p, width=700))